https://shendurelab.github.io/Lindel/docs/

States that Lindel is:

```
Compatible with both Python2.7 and Python3.5 or higher.

Requires biopython,numpy and scipy
```

Let's start with python 3.5

So we create both (to see if they work)

```bash
conda create -n lindel_py3 python=3.5 biopython numpy scipy
```

```bash
conda activate lindel_py3
```

We then navigate to /Lindel/ and run
```bash
python setup.py install
```


We get errors, one of them being:


```
Extracting Lindel-1.0-py3.5.egg to c:\users\tzion\miniconda3\envs\lindel_py3\lib\site-packages
  File "c:\users\tzion\miniconda3\envs\lindel_py3\lib\site-packages\Lindel-1.0-py3.5.egg\Lindel\Predictor.py", line 167
    "Sequence": f"{seq[:30]} | {seq[30:60]}",
                                           ^
SyntaxError: invalid syntax
```

The error we are seeing is because the code for Lindel uses f-strings (formatted string literals), which were introduced in Python 3.6.

We try again with python 3.6

```bash
conda create -n lindel_py3.6 python=3.6 biopython numpy scipy
```

```bash
conda activate lindel_py3.6
```

We then navigate to /Lindel/ and run
```bash
python setup.py install
```

Now it seems to work

Running
```bash
python Lindel_prediction.py -s TAACGTTATCAACGCCTATATTAAAGCGACCGTCGGTTGAACTGCGTGGATCAATGCGTC 
```

Seems to generate predictions:


Sequence: TAACGTTATCAACGCCTATATTAAAGCGACCGTCGGTTGAACTGCGTGGATCAATGCGTC  
Frameshift ratio: 0.8912  
Number of predictions: 20  

Top predictions:
 1. D4 at position -32 - 30.90%
     TAACGTTATCAACGCCTATATTAAAGCG-- | --TCGGTTGAACTGCGTGGATCAATGCGTC
 2. D1 at position -30 - 16.37%
     TAACGTTATCAACGCCTATATTAAAGCGAC | -GTCGGTTGAACTGCGTGGATCAATGCGTC
 3. I3+X - 8.77%
     TAACGTTATCAACGCCTATATTAAAGCGAC X CGTCGGTTGAACTGCGTGGATCAATGCGTC
 4. I1+C - 6.07%
     TAACGTTATCAACGCCTATATTAAAGCGAC C CGTCGGTTGAACTGCGTGGATCAATGCGTC
 5. D7 at position -32 - 3.96%
     TAACGTTATCAACGCCTATATTAAAGCG-- | -----GTTGAACTGCGTGGATCAATGCGTC
 6. D4 at position -30 - 2.18%
     TAACGTTATCAACGCCTATATTAAAGCGAC | ----GGTTGAACTGCGTGGATCAATGCGTC
 7. D8 at position -30 - 1.60%
     TAACGTTATCAACGCCTATATTAAAGCGAC | --------GAACTGCGTGGATCAATGCGTC
 8. D6 at position -30 - 1.35%
     TAACGTTATCAACGCCTATATTAAAGCGAC | ------TTGAACTGCGTGGATCAATGCGTC
 9. I1+A - 1.23%
     TAACGTTATCAACGCCTATATTAAAGCGAC A CGTCGGTTGAACTGCGTGGATCAATGCGTC
10. D7 at position -31 - 1.08%
     TAACGTTATCAACGCCTATATTAAAGCGA- | ------TTGAACTGCGTGGATCAATGCGTC

We add the `--top 0` argument to supress most of the output since the FS% doesn't change


```bash
python Lindel_prediction.py -s TAACGTTATCAACGCCTATATTAAAGCGACCGTCGGTTGAACTGCGTGGATCAATGCGTC --top 0
```

Results:  
Sequence: TAACGTTATCAACGCCTATATTAAAGCGACCGTCGGTTGAACTGCGTGGATCAATGCGTC  
Frameshift ratio: 0.8912  
Number of predictions: 0  

Top predictions:


Since this is working we can now move to interactive python supported version

```bash
# Create the environment with python 3.6 and interactive tools
conda create -n lindel_ipy python=3.6 biopython numpy scipy ipykernel ipython

# Activate the environment
conda activate lindel_ipy

# Register this environment as a Jupyter kernel
python -m ipykernel install --user --name lindel_ipy --display-name "Python 3.6 (Lindel)"
```

We now import the module to the notebook

In [1]:
import os
import sys
import io
import re
lindel_path = r'C:\Users\tzion\Documents\GitHub\Lindel'
if lindel_path not in sys.path:
    sys.path.append(lindel_path)
import Lindel_prediction


In [2]:
def get_frameshift_ratio(sequence):
    """
    Calls Lindel's main() and extracts only the Frameshift ratio value.
    Handles SystemExit and formatting errors gracefully.
    """
    # 1. Pre-validation: Lindel requires exactly 60bp
    if not isinstance(sequence, str) or len(sequence) != 60:
        print(f"Error: Sequence must be a 60bp string. (Got {len(str(sequence)) if sequence else 0}bp)")
        return None

    # 2. Back up environment
    original_argv = sys.argv
    original_stdout = sys.stdout
    captured_output = io.StringIO()
    
    try:
        # Redirect output and set arguments
        sys.stdout = captured_output
        sys.argv = ["Lindel_prediction.py", "-s", sequence, "--top", "0"]
        
        # 3. Call main (Catching SystemExit which Lindel uses for errors)
        try:
            Lindel_prediction.main()
        except SystemExit:
            # This happens if Lindel's internal validation fails
            pass 
            
        output_text = captured_output.getvalue()
        
        # 4. Extract ratio
        match = re.search(r"Frameshift ratio:\s*([\d\.]+)", output_text)
        
        if match:
            return float(match.group(1))
        else:
            # If we reach here, Lindel ran but didn't produce a ratio
            # Check for internal error messages in the captured text
            sys.stdout = original_stdout # Restore temporarily to print error
            print(f"--- Lindel Error Trace ---\n{output_text.strip()}\n-------------------------")
            return None
            
    except Exception as e:
        sys.stdout = original_stdout
        print(f"Unexpected Error: {e}")
        return None
        
    finally:
        # 5. Always restore environment
        sys.stdout = original_stdout
        sys.argv = original_argv


The program only works for 60BP sequences, and NOT 60-65 as documented on GitHub!

In [3]:
# Lets first test on a 60 BP sequence
my_seq = "TAACGTTATCAACGCCTATATTAAAGCGACCGTCGGTTGAACTGCGTGGATCAATGCGTC"
ratio = get_frameshift_ratio(my_seq)

if ratio is not None:
    print(f"Success! Frameshift Ratio: {ratio}")

# Now <60
my_seq = "TAACGTTATCAACGCCTATATTAAAGCGACCGTCGGTTGAACGCGTGGATCAATGCGTC"
ratio = get_frameshift_ratio(my_seq)

if ratio is not None:
    print(f"Success! Frameshift Ratio: {ratio}")

# Now >60bp
my_seq = "TAACGTTATCAACGCCTATATTAAAGCGACCGTCGGTTGAAACTGCGTGGATCAATGCGTC"
ratio = get_frameshift_ratio(my_seq)

if ratio is not None:
    print(f"Success! Frameshift Ratio: {ratio}")

Success! Frameshift Ratio: 0.8912
Error: Sequence must be a 60bp string. (Got 59bp)
Error: Sequence must be a 60bp string. (Got 61bp)


Update the env to also have pandas using 
`pip install pandas`

Get all appropriate datasets

In [4]:
import pandas as pd
import glob
import os
import sys
import io
import re
import time

# Setup paths
data_dir = '../data/'
output_dir = './Lindel_preds/'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)


def get_60bp_window(full_seq, pam_pos):
    """Checks if a sequence can provide a 60bp window around the cut site (pam_pos - 3)."""
    cut_site = pam_pos - 3
    start = cut_site - 30
    end = cut_site + 30
    if start >= 0 and end <= len(full_seq):
        return full_seq[start:end]
    # print(f"    Sequence too short for 60bp window at PAM pos {pam_pos}: {full_seq}")
    # print(f"      Required indices: {start} to {end}, Sequence length: {len(full_seq)}")
    # print(f"      Full sequence: {full_seq}")
    return None

# Step 1: Filter for 100% compatible datasets
valid_datasets = []
all_csv_files = glob.glob(os.path.join(data_dir, "*.csv"))

print("Validating datasets for 100% Lindel compatibility...")
for f in all_csv_files:
    temp_df = pd.read_csv(f)
    is_valid = True
    
    for _, row in temp_df.iterrows():
        if get_60bp_window(row['sequence'], int(row['PAM position'])) is None:
            is_valid = False
            print(f"    Incompatible sequence found: {row['sequence']} (PAM pos: {row['PAM position']})")
            break
            
    if is_valid:
        valid_datasets.append(f)
        print(f"  [KEEP] {os.path.basename(f)}")
    else:
        print(f"  [DISCARD] {os.path.basename(f)} - Contains incompatible sequences.")

if not valid_datasets:
    print("No datasets were 100% compatible. Exiting.")
    sys.exit()

Validating datasets for 100% Lindel compatibility...
  [KEEP] ALDIT_HAP1.csv
  [KEEP] ALDIT_Jurkat.csv
  [KEEP] ALDIT_Jurkat_DNTTKO.csv
  [KEEP] ALDIT_K562.csv
  [KEEP] ALDIT_K562_DNTTOE.csv
    Incompatible sequence found: CAATCCGTCTGTCTGTTCGAAGTAGTGTGTTACCTTTTGCGATCCGGAGTAGTTCTGGGCGTCAACTCGCTCGTTGGTT (PAM pos: 56)
  [DISCARD] FORECasT_BOB.csv - Contains incompatible sequences.
    Incompatible sequence found: CAATCCGTCTGTCTGTTCGAAGTAGTGTGTTACCTTTTGCGATCCGGAGTAGTTCTGGGCGTCAACTCGCTCGTTGGTT (PAM pos: 56)
  [DISCARD] FORECasT_CHO.csv - Contains incompatible sequences.
    Incompatible sequence found: CAATCCGTCTGTCTGTTCGAAGTAGTGTGTTACCTTTTGCGATCCGGAGTAGTTCTGGGCGTCAACTCGCTCGTTGGTT (PAM pos: 56)
  [DISCARD] FORECasT_HAP1.csv - Contains incompatible sequences.
    Incompatible sequence found: CAATCCGTCTGTCTGTTCGAAGTAGTGTGTTACCTTTTGCGATCCGGAGTAGTTCTGGGCGTCAACTCGCTCGTTGGTT (PAM pos: 56)
  [DISCARD] FORECasT_K562.csv - Contains incompatible sequences.
    Incompatible sequence found: CAATCCGTCT

In [5]:
def update_progress(current, total, start_time, prefix=""):
    bar_length = 25
    fraction = current / total
    elapsed_time = time.time() - start_time
    
    # Calculate ETA
    if current > 0:
        avg_time_per_item = elapsed_time / current
        remaining_items = total - current
        eta_seconds = int(avg_time_per_item * remaining_items)
        
        # Format time to MM:SS
        eta_str = f"{eta_seconds // 60:02d}:{eta_seconds % 60:02d}"
    else:
        eta_str = "--:--"

    arrow = int(fraction * bar_length) * '#'
    padding = (bar_length - len(arrow)) * ' '
    
    sys.stdout.write(f"\r{prefix} [{arrow}{padding}] {int(fraction*100)}% ({current}/{total}) ETA: {eta_str} ")
    sys.stdout.flush()

In [6]:
# Step 2: Extract unique 60bp sequences from the valid datasets
unique_lindel_seqs = set()
for f in valid_datasets:
    df = pd.read_csv(f)
    for _, row in df.iterrows():
        unique_lindel_seqs.add(get_60bp_window(row['sequence'], int(row['PAM position'])))

unique_list = list(unique_lindel_seqs)
print(f"\nTotal unique 60bp sequences to predict: {len(unique_list)}")

# Step 3: Predict and Cache
def get_ratio_from_lindel(sequence):
    """Captures the frameshift ratio from Lindel's stdout."""
    original_argv = sys.argv
    original_stdout = sys.stdout
    captured = io.StringIO()
    try:
        sys.stdout = captured
        sys.argv = ["Lindel_prediction.py", "-s", sequence, "--top", "0"]
        Lindel_prediction.main()
        output = captured.getvalue()
        match = re.search(r"Frameshift ratio:\s*([\d\.]+)", output)
        return float(match.group(1)) if match else None
    except:
        return None
    finally:
        sys.stdout = original_stdout
        sys.argv = original_argv

lindel_cache = {}
start_time = time.time()
for i, seq in enumerate(unique_list):
    lindel_cache[seq] = get_ratio_from_lindel(seq)
    # Reusing your update_progress function from the inDelphi script
    update_progress(i + 1, len(unique_list), start_time, prefix="Lindel")

# Step 4: Map and Save
print("\n\nSaving compatible datasets...")
for file_path in valid_datasets:
    file_name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    
    df['Lindel_FS_ratio'] = df.apply(
        lambda row: lindel_cache.get(get_60bp_window(row['sequence'], int(row['PAM position']))),
        axis=1
    )
    
    output_path = os.path.join(output_dir, file_name)
    df.to_csv(output_path, index=False)
    print(f"Saved: {file_name}")


Total unique 60bp sequences to predict: 105095
Lindel [#########################] 100% (105095/105095) ETA: 00:00 

Saving compatible datasets...
Saved: ALDIT_HAP1.csv
Saved: ALDIT_Jurkat.csv
Saved: ALDIT_Jurkat_DNTTKO.csv
Saved: ALDIT_K562.csv
Saved: ALDIT_K562_DNTTOE.csv
Saved: SPROUT_T.csv
Saved: SPROUT_T_CROTON_VERSION.csv
Saved: XCRISP_FORECasT_HAP1.csv
Saved: XCRISP_FORECasT_mESC.csv
Saved: XCRISP_FORECasT_TREX2.csv
